[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Column Types


## What you will be able to do

Choose a column type for money, for dates and times, for a fixed set of choices, for an identifier
and for a yes or no, and say which Python type each one takes and returns. Say what SQLite really
stores for each, and where that differs from what the Python value meant: the digits a `Numeric`
loses, the time zone a `DateTime` drops, the name an `Enum` stores. Change the types that annotations
choose, read a table's types on PostgreSQL and MySQL, and recognize the errors a string, an enum and
a missing length produce.


## The idea

### The problem

The college's bursar starts keeping payments in the database: an amount of money, the moment it was
received, how it was paid, a receipt number, and whether it was refunded. Every one of those is a
Python value with a precise meaning. `Decimal("1234.10")` is exactly one thousand two hundred
thirty-four dollars and ten cents, not a number close to it. `9:30 in the morning, Eastern time` is a
moment, the same one as 14:30 in UTC. A method of payment is one of three choices and nothing else.

SQLite has five kinds of storage: `NULL`, an integer, a floating point number, text and bytes. It has
no decimal, no date, no time zone, no set of choices and no identifier type, so every one of those
values has to become one of the five on the way in and come back as itself on the way out, and in
some cases it cannot. A `Decimal` stored as a floating point number is an approximation of itself,
and arithmetic on it in SQL shows it. A time with a zone is stored without the zone, silently, so a
payment made at 9:30 in New York comes back as 9:30 with no zone at all, five hours from where it
was.

### What a column type is

> A **column type** is the object that says what kind of value a column holds, and converts it: from
> the Python value to what the driver sends, and from what the database returns to the Python value.
> SQLAlchemy's generic types, such as **`Integer`**, **`String(n)`**, **`Numeric(p, s)`**,
> **`DateTime`**, **`Enum`**, **`Uuid`** and **`Boolean`**, become each database's own types: `Uuid`
> is `UUID` on PostgreSQL and 32 hexadecimal characters on SQLite. A `Mapped[...]` annotation chooses
> a type through the base's **type annotation map**: `Decimal` becomes `Numeric`, `datetime`
> becomes `DateTime`, an `enum.Enum` subclass becomes `Enum`, `uuid.UUID` becomes `Uuid`, and `bool`
> becomes `Boolean`, and **`type_annotation_map`** on the base changes any of them.

### Why it works that way

- **The type is a promise about the Python side.** Whatever SQLite stores, a `Numeric` column
  returns a `Decimal`, a `DateTime` a `datetime`, and an `Enum` a member of its enum, so the program
  works with the values it meant.
- **SQLite stores what it can.** A `Numeric` becomes a floating point number, a `DateTime` text in a
  fixed format that sorts in time order, an `Enum` the name of the member, a `Uuid` 32 hexadecimal
  characters, and a `Boolean` 0 or 1. The **Type Affinity** notebook of the **sqlite3, Deep Dive**
  guide explains how SQLite decides.
- **The conversion is strict where guessing would be wrong.** A `DateTime` on SQLite accepts a
  `datetime` or a `date` and refuses a string, since text such as `01/02/2026` could be either of two
  days.
- **What cannot be stored is dropped, without a word.** SQLite has no time zones, so the zone of an
  aware `datetime` is lost, and a `Numeric` keeps about fifteen significant digits, the most a
  floating point number holds.
- **Each database spells the types its own way.** PostgreSQL has an exact `NUMERIC`, a timestamp that
  keeps its zone, `UUID` and a real enum type, and MySQL insists on a length for every `VARCHAR`. A
  table that compiles for SQLite can fail to compile for MySQL.

### Where this shows up

Pydantic, in the **APIs and JSON** guide, converts the same Python types, `Decimal`, `datetime` and
`uuid.UUID`, at the edge of a web service, and SQLModel, in the **SQLModel, Deep Dive** guide, uses
one annotation for both. The **Everyday Requests** notebook of the **sqlite3, Deep Dive** guide kept
money in integer cents for exactly the reason this notebook shows, and its
**Adapters and Converters** notebook converted dates by hand, which is what a column type does for
you. **The Session** notebook adds `JSON`, the type whose changes a session can miss.

### What this notebook covers

- The annotations of a `Payment` class, and the types they choose
- What SQLite stores, and what comes back
- Money: `Numeric`, and arithmetic done by SQLite
- Times: storing a moment in UTC
- An enum stored by name
- Changing the defaults with `type_annotation_map`
- The same table on PostgreSQL
- Which type to use for which value
- The bursar's ledger, finished
- Five errors, from a date given as a string to a `VARCHAR` with no length

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from datetime import datetime
from decimal import Decimal

from sqlalchemy import Numeric, create_engine, select, text
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column


class Base(DeclarativeBase):
    pass


class Payment(Base):
    __tablename__ = "payments"
    id: Mapped[int] = mapped_column(primary_key=True)
    amount: Mapped[Decimal] = mapped_column(Numeric(10, 2))
    received_at: Mapped[datetime]


engine = create_engine("sqlite://")
Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add(Payment(amount=Decimal("1234.10"), received_at=datetime(2026, 1, 12, 9, 30)))
    payment = session.scalars(select(Payment)).one()
    print(repr(payment.amount), repr(payment.received_at))
    print(session.execute(text("SELECT typeof(amount), typeof(received_at) FROM payments")).one())
```

```
Decimal('1234.10') datetime.datetime(2026, 1, 12, 9, 30)
('real', 'text')
```

Python got back a `Decimal` and a `datetime`, and SQLite held a floating point number and text. The
column types did both conversions, and SQLite's `typeof()` shows what it really stored.


## Setup

Twelve imports, the college built from its `MetaData`, and a helper that prints a `CREATE TABLE`.

- `sqlalchemy` is the library itself, and the cell prints its version
- `Numeric`, `DateTime` and `Enum`, from `sqlalchemy`, are the column types this notebook names,
  with `select`, `insert`, `func` and `text`, `create_engine` and `event`, and what describes a
  table, `MetaData`, `Table`, `Column`, `Integer`, `String`, `Date`, `ForeignKey` and the
  constraints
- `DeclarativeBase`, `Mapped`, `mapped_column` and `Session`, from `sqlalchemy.orm`, map the classes
- `postgresql` and `mysql`, from `sqlalchemy.dialects`, write a table's SQL for those databases, with
  no server
- `CreateTable`, from `sqlalchemy.schema`, turns a table into its `CREATE TABLE` statement
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `Decimal`, `datetime`, `timezone`, `timedelta` and `date` are the values that money and moments
  are made of, `enum` defines the methods of payment, and `uuid` makes receipt numbers
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

Setup builds `scratch/college.db` as the **SQL Expressions** notebook did, and `show_ddl` is the
helper from the **Tables and Metadata** notebook, taking a dialect instead of an engine, so that it
can print a table for a database that has no engine here. This notebook adds a `payments` table to
the college.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import enum
import shutil
import uuid
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, Column, Date, DateTime, Enum, ForeignKey, Integer, MetaData, Numeric, String,
                        Table, UniqueConstraint, create_engine, event, func, insert, select, text)
from sqlalchemy.dialects import mysql, postgresql
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column
from sqlalchemy.pool import StaticPool
from sqlalchemy.schema import CreateTable

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}
college = MetaData(naming_convention=NAMING)

students = Table(
    "students", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True),
    Column("program", String(50), nullable=False),
    Column("started_on", Date, nullable=False),
)
courses = Table(
    "courses", college,
    Column("id", Integer, primary_key=True),
    Column("code", String(10), nullable=False, unique=True),
    Column("title", String(100), nullable=False),
    Column("department", String(50), nullable=False),
    Column("credits", Integer, nullable=False),
    CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),
)
terms = Table(
    "terms", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(20), nullable=False, unique=True),
    Column("starts_on", Date, nullable=False),
)
sections = Table(
    "sections", college,
    Column("id", Integer, primary_key=True),
    Column("course_id", ForeignKey("courses.id"), nullable=False),
    Column("term_id", ForeignKey("terms.id"), nullable=False),
    Column("capacity", Integer, nullable=False),
    UniqueConstraint("course_id", "term_id"),
    CheckConstraint("capacity > 0", name="capacity_positive"),
)
enrollments = Table(
    "enrollments", college,
    Column("student_id", ForeignKey("students.id"), primary_key=True),
    Column("section_id", ForeignKey("sections.id"), primary_key=True),
    Column("status", String(20), nullable=False, server_default="enrolled"),
    Column("grade", String(2)),
    CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),
)

def build_college(engine):
    """Create the college's tables from `college`, load the lists from Setup into them, and count their rows."""
    college.create_all(engine)
    rows = {
        students: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                   for name, email, program, started in STUDENTS],
        courses: [{"code": code, "title": title, "department": department, "credits": credits}
                  for code, title, department, credits in COURSES],
        terms: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        sections: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        enrollments: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                      for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for table in college.sorted_tables:
            conn.execute(insert(table), rows[table])
        return {table.name: conn.execute(select(func.count()).select_from(table)).scalar_one()
                for table in college.sorted_tables}

def show_ddl(table, dialect):
    """Print the CREATE TABLE statement a Table becomes in one database's dialect."""
    for line in str(CreateTable(table).compile(dialect=dialect)).strip().splitlines():
        print("   ", line.rstrip().replace("\t", "    "))

engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


## Worked examples

### The annotations of a Payment class, and the types they choose

`Method` is an enum of the three ways to pay. `Payment` annotates every column with the Python type
it holds, and adds a type with `mapped_column` only for the amount, whose precision the annotation
cannot say. `Student` maps just the two columns of `students` that this notebook reads, which a
mapped class is free to do:


In [2]:
class Method(enum.Enum):
    CARD = "card"
    TRANSFER = "transfer"
    CASH = "cash"


class Base(DeclarativeBase):
    pass


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))


class Payment(Base):
    __tablename__ = "payments"

    id: Mapped[int] = mapped_column(primary_key=True)
    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"))
    amount: Mapped[Decimal] = mapped_column(Numeric(10, 2))
    received_at: Mapped[datetime]
    method: Mapped[Method]
    receipt: Mapped[uuid.UUID]
    refunded: Mapped[bool] = mapped_column(default=False)
    note: Mapped[str | None]

    def __repr__(self):
        return f"Payment({self.amount}, {self.method.name}, {self.received_at:%Y-%m-%d %H:%M})"


for column in Payment.__table__.columns:
    print(f"{column.name:<12} {column.type!r}")
show_ddl(Payment.__table__, engine.dialect)
Base.metadata.create_all(engine)                  # creates payments; students is already there


id           Integer()
student_id   Integer()
amount       Numeric(precision=10, scale=2)
received_at  DateTime()
method       Enum('CARD', 'TRANSFER', 'CASH', name='method')
receipt      Uuid()
refunded     Boolean()
note         String()
    CREATE TABLE payments (
        id INTEGER NOT NULL,
        student_id INTEGER NOT NULL,
        amount NUMERIC(10, 2) NOT NULL,
        received_at DATETIME NOT NULL,
        method VARCHAR(8) NOT NULL,
        receipt CHAR(32) NOT NULL,
        refunded BOOLEAN NOT NULL,
        note VARCHAR,
        PRIMARY KEY (id),
        FOREIGN KEY(student_id) REFERENCES students (id)
    )


Every annotation chose a type: `Decimal` a `Numeric`, `datetime` a `DateTime`, `Method` an `Enum` of
its three names, `uuid.UUID` a `Uuid` and `bool` a `Boolean`. The `CREATE TABLE` is SQLite's version
of each. `method` became `VARCHAR(8)`, long enough for the longest name, `TRANSFER`, and `receipt`
became `CHAR(32)`. `NUMERIC(10, 2)` means ten digits, two of them after the point, which on SQLite is
only a name, as the next section shows.

### What SQLite stores, and what comes back

A payment goes in with Python values, and comes back twice: through the class, as Python values, and
through `text()`, as what SQLite actually stored, with `typeof()` naming its kind of storage. A
receipt number made by `uuid.uuid5` from a fixed name is the same on every run, where `uuid.uuid4`
is random:


In [3]:
first_receipt = uuid.uuid5(uuid.NAMESPACE_URL, "https://college.edu/receipts/1")

with Session(engine) as session:
    session.add(Payment(student_id=1, amount=Decimal("1234.10"), received_at=datetime(2026, 1, 12, 9, 30),
                        method=Method.CARD, receipt=first_receipt))
    session.commit()
    payment = session.scalars(select(Payment)).one()
    for name in ("amount", "received_at", "method", "receipt", "refunded", "note"):
        print(f"{name:<12} {getattr(payment, name)!r}")

with engine.connect() as conn:
    stored = conn.execute(text("SELECT amount, received_at, method, receipt, refunded FROM payments")).one()
    kinds = conn.execute(text("SELECT typeof(amount), typeof(received_at), typeof(method), typeof(receipt), "
                              "typeof(refunded) FROM payments")).one()
print("--- what SQLite stored:")
for name, kind, value in zip(stored._fields, kinds, stored):
    print(f"{name:<12} {kind:<8} {value!r}")


amount       Decimal('1234.10')
received_at  datetime.datetime(2026, 1, 12, 9, 30)
method       <Method.CARD: 'card'>
receipt      UUID('463e1d62-ca33-5522-8888-8ce4ab34ced7')
refunded     False
note         None
--- what SQLite stored:
amount       real     1234.1
received_at  text     '2026-01-12 09:30:00.000000'
method       text     'CARD'
receipt      text     '463e1d62ca33552288888ce4ab34ced7'
refunded     integer  0


Every value came back as the type its annotation named, `refunded` with its default, `False`, and
`note` as `None`. Underneath, the amount is a floating point number, `real`, the moment is text in a
format that sorts in time order, the method is the name `CARD` and not its value `card`, the receipt
is 32 hexadecimal digits with no hyphens, and `False` is the integer 0.

### Money: Numeric, and arithmetic done by SQLite

The `Numeric` column returned `Decimal('1234.10')` because SQLAlchemy rounds what SQLite stored to
the column's two places on the way out. Arithmetic done by SQLite itself gets no such help:


In [4]:
with engine.connect() as conn:
    print("amount * 3, worked out by SQLite:  ", conn.execute(text("SELECT amount * 3 FROM payments")).scalar_one())
print("Decimal('1234.10') * 3, by Python:  ", Decimal("1234.10") * 3)


amount * 3, worked out by SQLite:   3702.2999999999997
Decimal('1234.10') * 3, by Python:   3702.30


Three installments of 1,234.10 came to 3,702.2999999999997 in SQLite, because `1234.1` has no exact
binary floating point form, and 3,702.30 in Python's `Decimal`. On SQLite, keep money exact by doing
its arithmetic in Python with `Decimal`, or by storing whole cents in an `Integer` column, as the
**Everyday Requests** notebook of the **sqlite3, Deep Dive** guide does. PostgreSQL's `NUMERIC` is
exact, and does the same arithmetic without the error.

### Times: storing a moment in UTC

A time from a form usually carries its offset from UTC. SQLite cannot keep the offset, so the safe
habit is to turn every moment into UTC before it is stored, keep it without a zone, and put UTC back
on it when it is read:


In [5]:
paid = datetime.fromisoformat("2026-01-14T09:30-05:00")                  # 9:30 in New York, from a form
in_utc = paid.astimezone(timezone.utc)
print(paid, "->", in_utc, "-> stored as", in_utc.replace(tzinfo=None))

second_receipt = uuid.uuid5(uuid.NAMESPACE_URL, "https://college.edu/receipts/2")
with Session(engine) as session:
    session.add(Payment(student_id=1, amount=Decimal("850.00"), received_at=in_utc.replace(tzinfo=None),
                        method=Method.TRANSFER, receipt=second_receipt))
    session.commit()
    back = session.scalars(select(Payment).where(Payment.receipt == second_receipt)).one()
    print("read back:", repr(back.received_at), "-> in UTC:", back.received_at.replace(tzinfo=timezone.utc))
    print("in New York again:", back.received_at.replace(tzinfo=timezone.utc).astimezone(timezone(timedelta(hours=-5))))


2026-01-14 09:30:00-05:00 -> 2026-01-14 14:30:00+00:00 -> stored as 2026-01-14 14:30:00
read back: datetime.datetime(2026, 1, 14, 14, 30) -> in UTC: 2026-01-14 14:30:00+00:00
in New York again: 2026-01-14 09:30:00-05:00


`astimezone(timezone.utc)` moved 9:30 at five hours behind UTC to 14:30 in UTC, the same moment, and
the column stored 14:30 with no zone. Reading it back gave a naive `datetime`, and
`replace(tzinfo=...)` said what the program already knew, that every stored time is UTC. Converting
to a zone for display happens last, and only for the person reading. Common errors shows what
happens when the zone is left for SQLite to keep.

### An enum stored by name

`Payment.method` stores `CARD`, the member's name, and returns the member. A condition on the column
takes a member too, and SQLAlchemy writes its name into the SQL:


In [6]:
by_card = select(func.count()).select_from(Payment).where(Payment.method == Method.CARD)
print(by_card.compile(engine, compile_kwargs={"literal_binds": True}))
with Session(engine) as session:
    print(session.scalar(by_card), "payment by card |", [method.name for method in Method], "|", Method("card"))


SELECT count(*) AS count_1 
FROM payments 
WHERE payments.method = 'CARD'
1 payment by card | ['CARD', 'TRANSFER', 'CASH'] | Method.CARD


The SQL compares with `'CARD'`, the stored name, and `Method("card")` turns the value a form sends
into the member. `session.scalar()` returns the first column of the first row, like the result's
`scalar()`. Storing the name is SQLAlchemy's default, which matters when a table already holds the
values, as the college's `enrollments.status` does: Common errors has the error it produces and the
setting that changes it.

### Changing the defaults with type_annotation_map

A base's `type_annotation_map` changes the type any annotation chooses, for every class on that base.
A college that wants every string to have a length, every amount two decimal places and every time
with a zone says so once:


In [7]:
class TypedBase(DeclarativeBase):
    type_annotation_map = {str: String(100), Decimal: Numeric(10, 2), datetime: DateTime(timezone=True)}


class Fee(TypedBase):
    __tablename__ = "fees"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    amount: Mapped[Decimal]
    due: Mapped[datetime]


for column in Fee.__table__.columns:
    print(f"{column.name:<7} {column.type!r}")


id      Integer()
name    String(length=100)
amount  Numeric(precision=10, scale=2)
due     DateTime(timezone=True)


`Mapped[str]` became `String(length=100)`, `Mapped[Decimal]` a `Numeric` with two places, and
`Mapped[datetime]` a `DateTime` with `timezone=True`, with no `mapped_column` in the class. A column
that needs something else still says so with `mapped_column`, which wins over the map.

### The same table on PostgreSQL

The types are generic, so the same `Payment` compiles for other databases, with no server:


In [8]:
show_ddl(Payment.__table__, postgresql.dialect())


    CREATE TABLE payments (
        id SERIAL NOT NULL,
        student_id INTEGER NOT NULL,
        amount NUMERIC(10, 2) NOT NULL,
        received_at TIMESTAMP WITHOUT TIME ZONE NOT NULL,
        method method NOT NULL,
        receipt UUID NOT NULL,
        refunded BOOLEAN NOT NULL,
        note VARCHAR,
        PRIMARY KEY (id),
        FOREIGN KEY(student_id) REFERENCES students (id)
    )


PostgreSQL gets its own types: `TIMESTAMP WITHOUT TIME ZONE`, a real `UUID`, a real `BOOLEAN`, an
exact `NUMERIC(10, 2)`, and an enum type of its own, named `method`. The `id` became `SERIAL`,
PostgreSQL's automatically numbered integer. MySQL is stricter still, as the last of the Common
errors shows.

### Which type to use for which value

| Use | When | Why |
|---|---|---|
| `Numeric(p, s)`, arithmetic in Python with `Decimal` | money, on any database | exact in Python, and exact in the database too on PostgreSQL |
| `Integer` holding cents | money that SQLite must add up itself | whole numbers stay exact in SQL |
| `Float` | measurements, never money | fast, and approximate by design |
| `DateTime`, holding UTC | moments, on SQLite | SQLite keeps no zone, so the program keeps one convention |
| `DateTime(timezone=True)` | moments, on PostgreSQL | stored as a moment, and returned with its zone |
| `Enum(..., values_callable=...)` | a column whose stored text must be the members' values | the default stores their names |
| `Uuid` | identifiers that must be unique across systems | native on PostgreSQL, 32 characters on SQLite |
| `String(n)`, always with a length | text | MySQL requires the length, and the others use it as a limit |

The defaults: `Numeric` with Python doing the arithmetic, `DateTime` holding UTC, and `String(n)`
with a length, set once in `type_annotation_map`.

### The bursar's ledger, finished

The pieces of this notebook in two functions. `record_payment` takes what a form sends, all of it
text: an amount, rounded to cents as a `Decimal`, a time with its offset, turned into UTC, a method,
by its value, and a receipt number, turned into a `Uuid`. `statement` lists a student's payments
that were not refunded, oldest first, with their times in UTC, and works out what is still owed in
`Decimal`:


In [9]:
def record_payment(session, student_id, amount, paid_at, method, number):
    """Record a payment from a form's text: an amount, a time with its offset, a method, and a receipt number."""
    payment = Payment(
        student_id=student_id,
        amount=Decimal(amount).quantize(Decimal("0.01")),
        received_at=datetime.fromisoformat(paid_at).astimezone(timezone.utc).replace(tzinfo=None),
        method=Method(method),
        receipt=uuid.uuid5(uuid.NAMESPACE_URL, f"https://college.edu/receipts/{number}"),
    )
    session.add(payment)
    return payment


def statement(session, student_id, tuition):
    """A student's payments, oldest first, with their times in UTC, and the tuition still owed, in Decimal."""
    payments = session.scalars(
        select(Payment).where(Payment.student_id == student_id, Payment.refunded.is_(False)).order_by(Payment.received_at)
    ).all()
    lines = [(payment.received_at.replace(tzinfo=timezone.utc).isoformat(), payment.method.value, payment.amount)
             for payment in payments]
    return lines, tuition - sum((payment.amount for payment in payments), Decimal("0.00"))



with Session(engine) as session:
    record_payment(session, 1, "400", "2026-02-02T16:05+01:00", "cash", 3)
    refunded = record_payment(session, 1, "99.999", "2026-02-03T08:00+00:00", "card", 4)
    refunded.refunded = True
    session.commit()
    lines, owed = statement(session, 1, tuition=Decimal("3000.00"))

for when, method, amount in lines:
    print(f"{when:<26} {method:<9} {amount:>9}")
print(f"{'still owed':<36} {owed:>9}")


2026-01-12T09:30:00+00:00  card        1234.10
2026-01-14T14:30:00+00:00  transfer     850.00
2026-02-02T15:05:00+00:00  cash         400.00
still owed                              515.90


Ana Reyes has paid three times: the card payment from the first sections, the transfer from New York,
and 400 in cash at 16:05 one hour ahead of UTC, which is 15:05 in UTC. The fourth payment was
refunded, so the statement leaves it out, and its `99.999` was rounded to `100.00` by `quantize`
before it was ever stored. The total, 3,000.00 less 2,484.10, is exactly 515.90, because every step
was done in `Decimal`.

### Where each part came from

| In the ledger | What it relies on | The section that showed it |
|---|---|---|
| `Mapped[Decimal]` with `Numeric(10, 2)` | an amount returned as a `Decimal` | The annotations of a Payment class, and the types they choose |
| the total worked out in Python | `Decimal` arithmetic, exact where SQLite's is not | Money: Numeric, and arithmetic done by SQLite |
| `astimezone(timezone.utc)` before storing | a moment kept in UTC, since SQLite keeps no zone | Times: storing a moment in UTC |
| `Method(method)` | an enum member from the value a form sends | An enum stored by name |
| `uuid.uuid5(...)` | a receipt number that is a `Uuid`, the same on every run | What SQLite stores, and what comes back |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/08-column-types-solutions.ipynb).

**1.** Print the Python type that every column of `Payment` returns, with `column.type.python_type`.


In [10]:
# your code here


**2.** With `record_payment`, record a transfer of 500 from Ben Okafor, student 2, received at 15:45
on 20 January 2026, one hour ahead of UTC, and print its time as stored.


In [11]:
# your code here


**3.** Refund Ben Okafor's payment, and show with `typeof()` what SQLite now stores for `refunded`.


In [12]:
# your code here


**4.** In a database in memory, store `0.1 + 0.2` in a `Float` column and in a `Numeric(10, 2)`
column, and print what comes back from each.


In [13]:
# your code here


**5.** Map `Enrollment` with a `Status` enum whose stored text is the members' values, and count the
enrollments whose status is `Status.ENROLLED`.


In [14]:
# your code here


**6.** Print the `CREATE TABLE` of `Fee`, the class whose base has a `type_annotation_map`, for
PostgreSQL and for MySQL. What does each make of `due`, a `DateTime` with `timezone=True`?


In [15]:
# your code here


## Common errors

### sqlalchemy.exc.StatementError: (builtins.TypeError) SQLite DateTime type only accepts Python datetime and date objects as input.


In [16]:
NIGHTLY = {"student_id": 2, "amount": Decimal("75.00"), "method": Method.CASH,
           "receipt": uuid.uuid5(uuid.NAMESPACE_URL, "https://college.edu/receipts/5")}

with engine.begin() as conn:
    conn.execute(insert(Payment).values(**NIGHTLY, received_at="2026-01-15 10:00"))


StatementError: (builtins.TypeError) SQLite DateTime type only accepts Python datetime and date objects as input.
[SQL: INSERT INTO payments (student_id, amount, received_at, method, receipt, refunded) VALUES (?, ?, ?, ?, ?, ?)]

A nightly import copies payments from the cashier's system, and its times arrive as text. The
`DateTime` column refused the text rather than guess what it meant. SQLite would have stored the
string happily, and the column type is what stopped it. Turn text into a `datetime` first, with
`datetime.fromisoformat`, or `datetime.strptime` and the text's format:


In [17]:
with engine.begin() as conn:
    conn.execute(insert(Payment).values(**NIGHTLY, received_at=datetime.fromisoformat("2026-01-15 10:00")))
with Session(engine) as session:
    print(session.scalars(select(Payment).where(Payment.student_id == 2)).one())


Payment(75.00, CASH, 2026-01-15 10:00)


### No error, and five hours gone: a time zone that SQLite drops


In [18]:
class DeadlineBase(DeclarativeBase):
    pass


class Deadline(DeadlineBase):
    __tablename__ = "deadlines"

    id: Mapped[int] = mapped_column(primary_key=True)
    due: Mapped[datetime] = mapped_column(DateTime(timezone=True))


deadlines = college_engine()
DeadlineBase.metadata.create_all(deadlines)
new_york = timezone(timedelta(hours=-5))
with Session(deadlines) as session:
    session.add(Deadline(due=datetime(2026, 1, 30, 17, 0, tzinfo=new_york)))       # 17:00 in New York
    session.commit()
    due = session.scalars(select(Deadline.due)).one()
print("stored and read back:", repr(due))
print("the moment it meant: ", datetime(2026, 1, 30, 17, 0, tzinfo=new_york).astimezone(timezone.utc))
deadlines.dispose()


stored and read back: datetime.datetime(2026, 1, 30, 17, 0)
the moment it meant:  2026-01-30 22:00:00+00:00


`DateTime(timezone=True)` asks for a column that keeps the zone, and on SQLite there is no such
column. SQLAlchemy stored the clock time, 17:00, without the zone, and gave back a naive 17:00. Read
as UTC, as the rest of this notebook reads times, it is five hours earlier than the deadline meant,
and nothing raised. On SQLite, turn the moment into UTC before storing it, as the ledger does:


In [19]:
due_in_utc = datetime(2026, 1, 30, 17, 0, tzinfo=new_york).astimezone(timezone.utc).replace(tzinfo=None)
print("store this instead:", repr(due_in_utc))


store this instead: datetime.datetime(2026, 1, 30, 22, 0)


### No error, and fewer digits: a Numeric wider than SQLite can hold


In [20]:
class RateBase(DeclarativeBase):
    pass


class ExchangeRate(RateBase):
    __tablename__ = "exchange_rates"

    id: Mapped[int] = mapped_column(primary_key=True)
    rate: Mapped[Decimal] = mapped_column(Numeric(30, 20))


rates = college_engine()
RateBase.metadata.create_all(rates)
with Session(rates) as session:
    session.add(ExchangeRate(rate=Decimal("1.12345678901234567890")))
    session.commit()
    print("went in:   Decimal('1.12345678901234567890')")
    print("came back:", repr(session.scalars(select(ExchangeRate.rate)).one()))
rates.dispose()


went in:   Decimal('1.12345678901234567890')
came back: Decimal('1.12345678901234569125')


Twenty places went in, and every place after the sixteenth came back wrong: SQLite stored the value
as a floating point number, which holds about sixteen significant digits, and SQLAlchemy padded the
float out to the twenty places the column asked for. Nothing raised. A `Numeric` on SQLite is exact
only while the value fits in a float, which a two-place amount of money does. A value that needs
more belongs in text, or in a database with an exact `NUMERIC`, such as PostgreSQL.

### LookupError: 'completed' is not among the defined enum values. Enum name: status. Possible values: ENROLLED, COMPLETED, WITHDRAWN


In [21]:
class Status(enum.Enum):
    ENROLLED = "enrolled"
    COMPLETED = "completed"
    WITHDRAWN = "withdrawn"


class EnrollmentBase(DeclarativeBase):
    pass


class Enrollment(EnrollmentBase):
    __tablename__ = "enrollments"

    student_id: Mapped[int] = mapped_column(primary_key=True)
    section_id: Mapped[int] = mapped_column(primary_key=True)
    status: Mapped[Status]
    grade: Mapped[str | None] = mapped_column(String(2))


with Session(engine) as session:
    session.scalars(select(Enrollment).where(Enrollment.student_id == 1)).all()


LookupError: 'completed' is not among the defined enum values. Enum name: status. Possible values: ENROLLED, COMPLETED, WITHDRAWN

The college's `enrollments` table holds the values, `completed` and `enrolled`, and an `Enum` looks
rows up by the members' names, `COMPLETED` and `ENROLLED`, so the first row read could not be turned
into a member. The rows are fine and the mapping is wrong. `values_callable` tells the `Enum` which
text stands for each member, here the values, and `length` keeps the column wide enough for any
status:


In [22]:
class EnrollmentBase(DeclarativeBase):
    pass


class Enrollment(EnrollmentBase):
    __tablename__ = "enrollments"

    student_id: Mapped[int] = mapped_column(primary_key=True)
    section_id: Mapped[int] = mapped_column(primary_key=True)
    status: Mapped[Status] = mapped_column(Enum(Status, values_callable=lambda members: [m.value for m in members],
                                                length=20))
    grade: Mapped[str | None] = mapped_column(String(2))


with Session(engine) as session:
    first = session.scalars(select(Enrollment).where(Enrollment.student_id == 1).order_by(Enrollment.section_id)).first()
    print(first.section_id, repr(first.status), first.grade)


2 <Status.COMPLETED: 'completed'> C+


### sqlalchemy.exc.CompileError: (in table 'payments', column 'note'): VARCHAR requires a length on dialect mysql


In [23]:
show_ddl(Payment.__table__, mysql.dialect())


CompileError: (in table 'payments', column 'note'): VARCHAR requires a length on dialect mysql

`note: Mapped[str | None]` chose a `String` with no length, which SQLite and PostgreSQL accept as
text of any length and MySQL does not: a MySQL `VARCHAR` must say how long it can be. The error came
from compiling, with no server, which is the cheapest place to find it. Give the column a length, or
give every string one at once with `type_annotation_map`, as `Fee`'s base does:


In [24]:
copies = MetaData()                                       # copies of the tables, to change here
Student.__table__.to_metadata(copies)                     # the table the foreign key refers to
for_mysql = Payment.__table__.to_metadata(copies)
for_mysql.c.note.type = String(200)                       # a class would say mapped_column(String(200))
show_ddl(for_mysql, mysql.dialect())


    CREATE TABLE payments (
        id INTEGER NOT NULL AUTO_INCREMENT,
        student_id INTEGER NOT NULL,
        amount NUMERIC(10, 2) NOT NULL,
        received_at DATETIME NOT NULL,
        method ENUM('CARD','TRANSFER','CASH') NOT NULL,
        receipt CHAR(32) NOT NULL,
        refunded BOOL NOT NULL,
        note VARCHAR(200),
        PRIMARY KEY (id),
        FOREIGN KEY(student_id) REFERENCES students (id)
    )


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [25]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- An annotation chooses a column type through the base's type annotation map: `Decimal` a `Numeric`,
  `datetime` a `DateTime`, an enum an `Enum`, `uuid.UUID` a `Uuid`, `bool` a `Boolean`, and
  `type_annotation_map` changes the choices.
- The type returns the Python value it promised, whatever SQLite stored: a float for a `Numeric`,
  text for a `DateTime`, the member's name for an `Enum`.
- Money stays exact in Python's `Decimal`, and SQLite's own arithmetic on a `Numeric` is floating
  point.
- SQLite drops a time zone without a word, so moments go in as UTC.
- A `DateTime` refuses a string, an `Enum` stores names unless told otherwise, and MySQL needs every
  `String` to have a length.


## What is next

**The Session** notebook saves objects instead of reading them: `sessionmaker`, `add`, `flush` and
`commit`, and what autoflush does behind your back.


---

&#8592; **Previous:** [Declarative Models](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/07-declarative-models.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [The Session](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/09-the-session.ipynb) &#8594;
